# 07 · 卷积神经网络（CNN）与 CIFAR-10 图像分类

> CNN 是计算机视觉的基石。这一节从卷积原理讲到完整分类网络。

## 7.1 为什么需要卷积

全连接网络处理图像的三大问题：
1. **参数爆炸**：一张 224×224×3 的图展平是 150528 维，接一个 1000 维全连接层就是 1.5 亿参数。
2. **丢失空间结构**：展平后相邻像素关系消失。
3. **没有平移不变性**：图像平移后输出完全改变。

CNN 通过三个特性解决：
- **局部连接**：每个神经元只看一小块区域（感受野）。
- **权值共享**：同一个卷积核在整张图上滑动，参数量骤减。
- **平移等变**：特征在哪儿出现都能被检测到。

## 7.2 卷积运算

给定输入 $X$ 和卷积核 $K$（kernel/filter），输出：

$$
Y[i,j] = \sum_{m}\sum_{n} X[i+m,\ j+n]\cdot K[m,n] + b
$$

**输出尺寸公式**（单维）：

$$
H_{out} = \left\lfloor \frac{H_{in} + 2P - D(K-1) - 1}{S} \right\rfloor + 1
$$

- $P$ padding（补边）、$K$ kernel（核大小）、$S$ stride（步长）、$D$ dilation（膨胀）

常见配置：`K=3, P=1, S=1` → 尺寸不变；`K=3, P=0, S=1` → 尺寸 -2。

### 参数量

一层卷积参数 = $K_h \times K_w \times C_{in} \times C_{out} + C_{out}$（含 bias）。
**与图像大小无关！** 这就是权值共享的威力。

In [ ]:
import torch
import torch.nn as nn

# 手动实现 2D 卷积（理解用，实际用 nn.Conv2d）
def conv2d_manual(x, kernel, bias=0.0, stride=1, padding=0):
    if padding > 0:
        x = torch.nn.functional.pad(x, (padding,) * 4)
    C, H, W = x.shape
    kh, kw = kernel.shape
    oh = (H - kh) // stride + 1
    ow = (W - kw) // stride + 1
    out = torch.zeros(oh, ow)
    for i in range(oh):
        for j in range(ow):
            patch = x[:, i*stride:i*stride+kh, j*stride:j*stride+kw]
            out[i, j] = (patch * kernel).sum() + bias
    return out

x = torch.arange(25, dtype=torch.float32).reshape(1, 5, 5)
k = torch.tensor([[1., 0., -1.], [1., 0., -1.], [1., 0., -1.]])   # 竖直边缘检测
print("输入:\n", x[0])
print("\n卷积核(竖直边缘):\n", k)
print("\n手动卷积输出:\n", conv2d_manual(x, k))

# 用 nn.Conv2d 验证
conv = nn.Conv2d(1, 1, 3, bias=False)
with torch.no_grad():
    conv.weight.copy_(k.view(1, 1, 3, 3))
print("\nnn.Conv2d 输出:\n", conv(x.unsqueeze(0)).squeeze())

## 7.3 nn.Conv2d 参数详解

```python
nn.Conv2d(in_channels, out_channels, kernel_size,
          stride=1, padding=0, dilation=1, groups=1, bias=True)
```

In [ ]:
x = torch.randn(1, 3, 32, 32)      # (N, C, H, W)

# 基础卷积
c = nn.Conv2d(3, 16, 3, padding=1)
print("padding=1, k=3 :", c(x).shape)          # 尺寸不变

# 无 padding，尺寸减小
print("padding=0, k=3 :", nn.Conv2d(3, 16, 3)(x).shape)   # 30x30

# stride=2，下采样一半
print("stride=2       :", nn.Conv2d(3, 16, 3, stride=2, padding=1)(x).shape)  # 16x16

# 5x5 核
print("k=5, p=2       :", nn.Conv2d(3, 16, 5, padding=2)(x).shape)

In [ ]:
# 参数量对比：Conv vs Linear
c = nn.Conv2d(3, 64, 3, padding=1)
print("Conv2d(3->64,3x3) 参数:", sum(p.numel() for p in c.parameters()))

fc = nn.Linear(3 * 32 * 32, 64)
print("Linear(3072->64)  参数:", sum(p.numel() for p in fc.parameters()))
# 卷积少几十倍，且输入图再大参数也不变:
c2 = nn.Conv2d(3, 64, 3, padding=1)
print("输入 256x256 时 Conv 参数不变:", sum(p.numel() for p in c2.parameters()))
print("输入 256x256 时 Linear 参数:", sum(p.numel() for p in nn.Linear(3*256*256, 64).parameters()))

### groups 参数（分组卷积）

- `groups=1`：普通卷积
- `groups=in_channels`：**深度可分离卷积**（MobileNet 核心），参数量除以 in_channels
- `groups=g`：分成 g 组各自卷积

In [ ]:
x = torch.randn(1, 32, 16, 16)
normal = nn.Conv2d(32, 64, 3, padding=1)
depthwise = nn.Conv2d(32, 32, 3, padding=1, groups=32)
print("普通卷积参数:", sum(p.numel() for p in normal.parameters()))
print("深度卷积参数:", sum(p.numel() for p in depthwise.parameters()))

## 7.4 池化层

| 层 | 作用 |
|---|---|
| `nn.MaxPool2d(k)` | 取窗口最大值，保留最强响应 |
| `nn.AvgPool2d(k)` | 取平均值 |
| `nn.AdaptiveAvgPool2d(1)` | 全局平均池化，输出 1×1（替代 Flatten） |
| `nn.AdaptiveMaxPool2d` | 自适应最大池化 |

池化**无参数**，只做下采样，减少计算量并增大感受野。

In [ ]:
x = torch.randn(1, 1, 4, 4)
print("输入:\n", x[0, 0])
print("\nMaxPool2d(2):\n", nn.MaxPool2d(2)(x)[0, 0])
print("\nAvgPool2d(2):\n", nn.AvgPool2d(2)(x)[0, 0])
print("\nAdaptiveAvgPool2d(1):\n", nn.AdaptiveAvgPool2d(1)(torch.randn(2, 8, 7, 7)).shape)

### 用卷积替代池化（现代趋势）

stride=2 的卷积也能下采样，且可学习。ResNet 常用。

## 7.5 感受野（Receptive Field）

感受野 = 输出上一个像素"看到"的输入区域大小。

多层小卷积核比单层大卷积核更好：
- 两个 3×3 堆叠感受野 = 5×5，参数 = 2×9 = 18 < 25
- 三个 3×3 堆叠感受野 = 7×7，参数 = 3×9 = 27 < 49
- 且层数多 → 非线性更强

计算感受野：$RF_{out} = RF_{in} + (K-1)\times \text{stride}_{cum}$

In [ ]:
def receptive_field(layers):
    # layers: [(kernel, stride), ...]
    rf, stride = 1, 1
    for k, s in layers:
        rf = rf + (k - 1) * stride
        stride *= s
    return rf

print("两层3x3:", receptive_field([(3, 1), (3, 1)]))          # 5
print("三层3x3:", receptive_field([(3, 1), (3, 1), (3, 1)]))  # 7
print("一层7x7:", receptive_field([(7, 1)]))                  # 7
print("conv3x3+pool2:", receptive_field([(3, 1), (2, 2)]))

## 7.6 实战：CIFAR-10 分类

CIFAR-10：10 类彩色图片，32×32，训练 50000 张。

### 数据准备（含数据增强）

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import time

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

class SyntheticImageDataset(torch.utils.data.Dataset):
    # 当 CIFAR-10 无法下载时的备用图像数据（10 类，32x32）
    def __init__(self, n=3000, num_classes=10, size=32, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.data = torch.rand(n, 3, size, size, generator=g) * 0.5 - 0.25
        self.targets = torch.randint(0, num_classes, (n,), generator=g)
        for i in range(n):
            self.data[i, self.targets[i] % 3] += 0.7
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, i):
        return self.data[i], int(self.targets[i])


CLASSES = ['plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

try:
    train_set = datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
    test_set = datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
    USE_CIFAR = True
    print("使用真实 CIFAR-10")
except Exception as e:
    print("CIFAR-10 下载失败（可能网络较慢），改用合成图像数据继续演示：",
          type(e).__name__)
    train_set = SyntheticImageDataset(3000, seed=0)
    test_set = SyntheticImageDataset(500, seed=1)
    USE_CIFAR = False

print("训练:", len(train_set), "测试:", len(test_set))

In [ ]:
BATCH = 128
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

# 可视化
def denorm(t, mean=CIFAR_MEAN, std=CIFAR_STD):
    m = torch.tensor(mean).view(3, 1, 1)
    s = torch.tensor(std).view(3, 1, 1)
    return (t * s + m).clamp(0, 1)

imgs, labs = next(iter(DataLoader(train_set, batch_size=16, shuffle=True)))
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, im, la in zip(axes.ravel(), imgs, labs):
    ax.imshow(denorm(im).permute(1, 2, 0))
    ax.set_title(CLASSES[la]); ax.axis("off")
plt.suptitle("CIFAR-10 样本")
plt.tight_layout(); plt.show()

### 定义 CNN（VGG 风格）

结构：`Conv-BN-ReLU` 堆叠 + 池化下采样 + 全局平均池化 + 全连接。

In [ ]:
def conv_bn_relu(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )

class VGGSmall(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_bn_relu(3, 64),  conv_bn_relu(64, 64),   nn.MaxPool2d(2),   # 32->16
            conv_bn_relu(64, 128), conv_bn_relu(128, 128), nn.MaxPool2d(2),  # 16->8
            conv_bn_relu(128, 256), conv_bn_relu(256, 256), nn.MaxPool2d(2), # 8->4
            conv_bn_relu(256, 512), nn.MaxPool2d(2),                          # 4->2
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

model = VGGSmall().to(device)
print("参数量:", f"{sum(p.numel() for p in model.parameters()):,}")

with torch.no_grad():
    print("输出:", model(torch.randn(4, 3, 32, 32).to(device)).shape)

## 7.7 可视化卷积核与特征图

理解 CNN "看到了什么"。浅层学边缘/颜色，深层学纹理/物体部件。

In [ ]:
# 第一层卷积核可视化
w = model.features[0][0].weight.detach().cpu()      # (64, 3, 3, 3)
print("第一层卷积核形状:", w.shape)

fig, axes = plt.subplots(8, 8, figsize=(10, 10))
w_norm = (w - w.min()) / (w.max() - w.min())
for ax, ker in zip(axes.ravel(), w_norm):
    ax.imshow(ker.permute(1, 2, 0))
    ax.axis("off")
plt.suptitle("第一层 64 个卷积核（放大显示）")
plt.tight_layout(); plt.show()

In [ ]:
# 特征图可视化：取一张图，用 forward hook 抓取每个 Conv2d 的输出
img, label = test_set[0]
x = img.unsqueeze(0).to(device)
print("输入类别:", CLASSES[label])

activations = []

def hook(module, inp, out):
    activations.append(out.detach().cpu())

handles = [m.register_forward_hook(hook)
           for m in model.modules() if isinstance(m, nn.Conv2d)]

model.eval()
with torch.no_grad():
    model(x)

for h in handles:
    h.remove()

print(f"共捕获 {len(activations)} 个卷积层，形状:",
      [tuple(a.shape[1:]) for a in activations])

fig, axes = plt.subplots(len(activations), 8, figsize=(14, 2 * len(activations)))
if len(activations) == 1:
    axes = axes[None, :]
for row, act in enumerate(activations):
    for col in range(8):
        fmap = act[0, col]
        axes[row, col].imshow(fmap, cmap="viridis")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(f"conv{row}\n{act.shape[1]}ch")
plt.suptitle("不同卷积层的特征图（每层取前 8 个通道）")
plt.tight_layout(); plt.show()

## 7.8 训练

> **注意**：CIFAR-10 + 这个网络在 CPU 上训练较慢。下面会根据设备自动设定 epoch 数：
> 有 GPU 且用真实数据时跑 30 epoch，否则只跑 5 epoch 演示流程。

## 7.8 训练

In [ ]:
EPOCHS = 30 if (USE_CIFAR and device.type == "cuda") else 5
print("本次 EPOCHS =", EPOCHS, "（CPU 或备用数据时自动减少）")

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9,
                            weight_decay=5e-4, nesterov=True)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=0.1, epochs=EPOCHS, steps_per_epoch=len(train_loader))


def run_epoch(model, loader, train=True):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
        if train:
            loss.backward()
            optimizer.step()
            scheduler.step()
        total_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += y.size(0)
    return total_loss / n, correct / n


history = {"tl": [], "ta": [], "vl": [], "va": []}
best_acc = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tl, ta = run_epoch(model, train_loader, train=True)
    vl, va = run_epoch(model, test_loader, train=False)
    history["tl"].append(tl); history["ta"].append(ta)
    history["vl"].append(vl); history["va"].append(va)
    best_acc = max(best_acc, va)
    if epoch % 2 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{EPOCHS} | train {tl:.3f}/{ta:.3f} | "
              f"test {vl:.3f}/{va:.3f} | {time.time()-t0:.0f}s")

print(f"\n最佳测试准确率: {best_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history["tl"], label="train"); plt.plot(history["vl"], label="test")
plt.legend(); plt.title("Loss"); plt.grid(alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot(history["ta"], label="train"); plt.plot(history["va"], label="test")
plt.legend(); plt.title("Accuracy"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7.9 用预训练特征提取（快速达到高精度）

若不想从头训练，可以冻结卷积层、只训分类头（第 10 节详解）：
```python
model.features.requires_grad_(False)
```
因为我们的网络不是预训练权重，这里不演示，放到迁移学习一节。

## 7.10 数据增强消融实验（体会作用）

分别用"无增强"和"强增强"训练少量 epoch，比较验证准确率。

In [ ]:
def quick_train(train_tf, epochs=3, bs=128):
    if USE_CIFAR:
        ds = datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
        ds = Subset(ds, range(3000))          # 只用子集，加速演示
    else:
        ds = SyntheticImageDataset(2000, seed=0)
    loader = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=0)
    m = VGGSmall().to(device)
    opt = torch.optim.SGD(m.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        m.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(m(x), y)
            loss.backward(); opt.step()
    # 评估
    m.eval()
    correct = n = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (m(x).argmax(1) == y).sum().item(); n += y.size(0)
    return correct / n

plain = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])

print("无增强 3 epoch:", round(quick_train(plain, 3), 4))
print("强增强 3 epoch:", round(quick_train(aug, 3), 4))

## 7.11 练习

**练习 1**：手推：输入 32×32，经过 `Conv(3→64,k=3,p=1) → MaxPool2 → Conv(64→128,k=3,p=1) → MaxPool2`，
输出的空间尺寸是多少？用代码验证。

**练习 2**：把 `VGGSmall` 的全局平均池化换成 `Flatten + Linear(512*2*2, 10)`，比较参数量。

**练习 3**：把 `MaxPool2d` 换成 `Conv2d(stride=2)`，观察准确率变化。

**练习 4**：统计 CIFAR-10 每个类别的准确率，哪个类最难？

**练习 5**：把 `padding=1` 的 3×3 卷积换成 `padding=0`，观察尺寸变化和准确率下降。

In [ ]:
# 练习 1
import torch.nn as nn
x = torch.randn(1, 3, 32, 32)
net = nn.Sequential(
    nn.Conv2d(3, 64, 3, padding=1), nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3, padding=1), nn.MaxPool2d(2))
print("练习1 输出尺寸:", net(x).shape)      # (1,128,8,8)

# 练习 2
class VGGFC(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = VGGSmall().features
        self.c = nn.Sequential(nn.Flatten(), nn.Linear(512 * 2 * 2, 10))
    def forward(self, x):
        return self.c(self.f(x))
print("练习2 参数量:", f"{sum(p.numel() for p in VGGFC().parameters()):,}")

# 练习 4（需训练完成的 model）
with torch.no_grad():
    all_pred, all_lab = [], []
    for x, y in test_loader:
        all_pred.append(model(x.to(device)).argmax(1).cpu()); all_lab.append(y)
pred = torch.cat(all_pred); lab = torch.cat(all_lab)
print("\n练习4 各类准确率:")
for c in range(10):
    mask = lab == c
    print(f"  {CLASSES[c]:6s}: {(pred[mask] == lab[mask]).float().mean():.4f}")

## 7.12 小结

- 卷积 = 局部连接 + 权值共享；输出尺寸用公式 `(H+2P-K)/S + 1`。
- 标准块：`Conv3x3 + BN + ReLU`，池化/stride 下采样，`AdaptiveAvgPool` + `Linear` 分类。
- 对比全连接：卷积参数量与图像大小无关，空间结构保留。
- 现代趋势：小卷积核堆叠、全局平均池化、残差连接（第 11 节）。
- 数据增强显著提升泛化能力。

下一节：`08_序列模型RNN_LSTM_GRU.ipynb`